# Reevaluación de los Ejecutores con 150 semillas de evaluación (ABIDES)

**No entrena nada.** Carga las redes que ya se entrenaron (sus `checkpoint_last.pt` en Drive) y las evalúa, junto con la red inicial de cada semilla (control) y las tres heurísticas, en las **mismas 150 semillas**. Hace falta porque con 15 semillas el ruido del precio tapaba la diferencia entre estrategias y con 40 una conclusión cambió de signo (ver `docs/v4_abides_resultado_parcial_A_PS.md`).

Ejecuta las celdas **en orden** (o "Ejecutar todas"; si la celda 1 corta la ejecución, vuelve a lanzar "Ejecutar todas"). Dura **entre 60 y 120 minutos** según cuántas redes haya. Es reanudable: si Colab se corta o aparecen redes nuevas (las del cuaderno B de V4), vuelve a ejecutar la celda de evaluación y solo hace lo que falta. Todo se guarda en Drive en **`slippage_colab/resultados/reeval_150`** y al final se arma `reeval_150.zip`.

## Preparación del entorno (celdas 1 a 9)

In [ ]:
# CELDA 1 (reinicia el entorno solo; el aviso "sesión fallada" es normal, no la repitas)
!pip install -q "condacolab<0.2"
import condacolab
condacolab.install()

In [ ]:
# CELDA 2
import os
if not os.path.exists('/usr/local/envs/abides'):
    !conda create -n abides python=3.9 -y -q
!conda run -n abides python --version

In [ ]:
# CELDA 3 (unos minutos; un aviso de Pillow es normal)
%env PIP_DISABLE_PIP_VERSION_CHECK=1
!conda run -n abides python -m pip install -q pip==23.3.2 setuptools==57.5.0 wheel==0.38.4
!conda run -n abides python -m pip install -q "cython<3" numpy==1.22.0
!conda run -n abides python -m pip install -q --no-build-isolation --no-deps gym==0.18.0
!conda run -n abides python -m pip install -q --no-build-isolation numpy==1.22.0 "cloudpickle>=1.2,<1.7" "pyglet>=1.4,<=1.5.0" "Pillow<10" pandas==1.2.4 scipy==1.10.0 coloredlogs==15.0.1 psutil==5.8.0 tqdm==4.61.1 pomegranate==0.14.5 "ray[tune]==1.7.0" "protobuf==3.20.3" pyarrow==11.0.0 matplotlib==3.5.3 pytest
!conda run -n abides python -m pip install -q "gymnasium==0.29.1" "importlib-metadata>=4.8" farama-notifications --no-deps

In [ ]:
# CELDA 4 (deben salir tres "OK")
import os
if not os.path.exists('/content/abides-jpmc-public'):
    !git clone -q https://github.com/jpmorganchase/abides-jpmc-public.git /content/abides-jpmc-public
for pkg in ['abides-core', 'abides-markets', 'abides-gym']:
    !cd /content/abides-jpmc-public/{pkg} && conda run -n abides python setup.py -q install > /content/install_{pkg}.log 2>&1 && echo "{pkg} OK" || (echo "{pkg} FALLO"; tail -20 /content/install_{pkg}.log)

In [ ]:
# CELDA 5 (debe imprimir 0.18.0 1.22.0 1.2.4 0.29.1 1.7.0)
!conda run -n abides python -c "import gym, numpy, pandas, gymnasium, ray, abides_core, abides_markets, abides_gym; print(gym.__version__, numpy.__version__, pandas.__version__, gymnasium.__version__, ray.__version__)"

In [ ]:
# CELDA 6 (torch; debe imprimir torch 1.13.1+cpu | numpy 1.22.0)
!conda run -n abides python -m pip install -q torch==1.13.1+cpu --extra-index-url https://download.pytorch.org/whl/cpu
!conda run -n abides python -c "import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__)"

In [ ]:
# CELDA 7 (código del proyecto: rama con el reevaluador; debe mostrar un commit y un 1)
import os
if not os.path.exists('/content/slippage'):
    !git clone -q https://github.com/Vatelfil/slippage.git /content/slippage
!cd /content/slippage && git fetch -q origin && git checkout -q -f origin/feature/ps-abides-v4-resultados-A
!cd /content/slippage && git log --oneline -1 && ls scripts/colab/reevaluar_ejecutores_ps.py | wc -l

In [ ]:
# CELDA 8 (conecta Drive; acepta el permiso)
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/slippage_colab/resultados'
import os; os.makedirs(RESULTS_DIR, exist_ok=True)
print(RESULTS_DIR)

In [ ]:
# CELDA 9 (seaborn, que necesita el módulo de benchmarks)
!conda run -n abides python -m pip install -q seaborn==0.12.2 --no-deps

## Reevaluación

In [ ]:
# Antes de evaluar: lista las redes entrenadas que hay en Drive (cada una termina en checkpoint_last.pt)
!find {RESULTS_DIR} -name checkpoint_last.pt | sort

In [ ]:
# Reevaluación: 150 semillas, control, heurísticas y todas las redes encontradas (≈ 60 a 120 min; reanudable)
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/reevaluar_ejecutores_ps.py --raices {RESULTS_DIR}/conf_pen30_g1_A {RESULTS_DIR}/conf_pen30_g1_B {RESULTS_DIR}/v4_pen30_g1_A {RESULTS_DIR}/v4_pen30_g1_B {RESULTS_DIR}/d3_1000ep --n-semillas 150 --out-dir {RESULTS_DIR}/reeval_150 --max-minutes 150 2>&1 | grep -v UserWarning

## Zip final

In [ ]:
# CELDA FINAL: revisa lo que se guardó y arma un zip pequeño para descargar
import os, zipfile
base = f'{RESULTS_DIR}/reeval_150'
archivos = [os.path.join(base, a) for a in sorted(os.listdir(base)) if a.endswith('.json')]
print(len(archivos), 'archivos en', base)
for a in archivos: print(' ', os.path.basename(a))
zpath = f'{RESULTS_DIR}/reeval_150.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for a in archivos:
        z.write(a, os.path.join('reeval_150', os.path.basename(a)))
print('zip listo:', zpath, os.path.getsize(zpath), 'bytes  -> descárgalo desde slippage_colab/resultados en Drive')